# Figure 2 Reproduction — Brownian Motion vs Gradient Descent

This notebook is a self-contained version of the previous runner notebook.  It does **not** import experiment helpers from project `.py` files.  The model, true-label MNIST MDP, replay buffer, Q-learning loop, Brownian update, probe objective, gradient covariance, Hessian-vector products, Lanczos ESD approximation, and plotting code are all defined directly in notebook cells.

The experiment follows the Figure 2 / Section 4.2 idea from *Understanding Plasticity in Neural Networks*: compare two parameter trajectories that start from the same initialization. One trajectory follows SGD on a non-stationary Q-learning objective; the other receives random Brownian perturbations with the same update norm. Geometry is then measured using the probe loss

$$\ell(\theta) = \|f_\theta(X) - \mathrm{stopgrad}(f_\theta(X) + \epsilon)\|^2,$$

where $\epsilon \sim \mathcal{N}(0, 1)$.

## 1. Setup

This cell imports only standard scientific Python libraries and sets the output/data directories.  No project-level modules are imported.

In [1]:
from __future__ import annotations

import copy
import math
import os
import random
from collections import deque
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Deque, Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import KMeans
from torchvision import datasets, transforms
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
DATA_ROOT = ROOT / "data"
OUTPUT_ROOT = ROOT / "outputs"
OUTPUT_ROOT.mkdir(exist_ok=True)

MPLCONFIGDIR = OUTPUT_ROOT / "matplotlib_cache"
MPLCONFIGDIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

/Users/mahdigheidi/Documents/Univ/Masters-Study-Project/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cpu')

## 2. Configuration

The default values below are intentionally smoke-scale so the notebook can be tested quickly.  For a heavier reproduction, increase `hidden_dim`, `prefill_steps`, `train_steps`, `probe_batch_size`, `cov_batch_size`, and turn on `compute_hessian`.

In [ ]:
@dataclass
class Figure2Config:
    seed: int = 0
    data_root: str = str(DATA_ROOT)
    download: bool = False

    # Model and Q-learning settings
    hidden_dim: int = 128
    gamma: float = 0.99
    lr: float = 1e-3
    batch_size: int = 128
    replay_capacity: int = 10_000
    prefill_steps: int = 1_000
    train_steps: int = 1_000
    target_update_period: int = 500
    epsilon: float = 0.1
    online_collection_per_step: int = 10

    # Probe / geometry settings
    probe_batch_size: int = 128
    cov_batch_size: int = 128
    lanczos_iter: int = 20
    lanczos_vectors: int = 1
    esd_points: int = 400
    esd_sigma: Optional[float] = None
    snapshot_target_updates: Tuple[int, ...] = (1, 2)
    compute_hessian: bool = False

    # Plot settings
    esd_xlim: Tuple[float, float] = (0.0, 50.0)
    esd_ylim: Tuple[float, float] = (0.0, 0.4)
    covariance_vmin: float = -1.0
    covariance_vmax: float = 1.0
    output_path: str = str(OUTPUT_ROOT / "figure2_self_contained.png")


PAPER_SCALE_OVERRIDES = {
    "hidden_dim": 1024,
    "prefill_steps": 50_000,
    "train_steps": 100_000,
    "batch_size": 512,
    "probe_batch_size": 512,
    "cov_batch_size": 512,
    "target_update_period": 5_000,
    "online_collection_per_step": 100,
    "lanczos_iter": 100,
    "lanczos_vectors": 3,
    "compute_hessian": True,
}

cfg = Figure2Config()
asdict(cfg)

## 3. Reproducibility Utilities

This helper fixes the random seeds used by Python, NumPy, and PyTorch.

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## 4. True-Label MNIST Classification MDP

The latent state is one of the digits `{0, ..., 9}`.  Given state `s`, the environment returns an MNIST image with label `s`.  The reward is `1` when the selected action equals the latent state and `0` otherwise.  The next state is sampled uniformly.

In [ ]:
def load_true_label_mnist(data_root: str, download: bool) -> datasets.MNIST:
    transform = transforms.ToTensor()
    return datasets.MNIST(root=data_root, train=True, download=download, transform=transform)


class EasyMDP:
    def __init__(self, dataset: datasets.MNIST, seed: int = 0):
        self.dataset = dataset
        self.rng = random.Random(seed)
        self.indices_by_label = {i: [] for i in range(10)}

        targets = dataset.targets.tolist() if hasattr(dataset.targets, "tolist") else list(dataset.targets)
        for idx, label in enumerate(targets):
            self.indices_by_label[int(label)].append(idx)

        missing = [k for k, v in self.indices_by_label.items() if len(v) == 0]
        if missing:
            raise ValueError(f"Dataset is missing labels: {missing}")

        self.state = self.rng.randrange(10)

    def reset(self, state: Optional[int] = None) -> torch.Tensor:
        self.state = self.rng.randrange(10) if state is None else int(state)
        return self.sample_observation(self.state)

    def sample_observation(self, state: int) -> torch.Tensor:
        idx = self.rng.choice(self.indices_by_label[int(state)])
        image, _ = self.dataset[idx]
        return image

    def step(self, action: int) -> Tuple[torch.Tensor, float, int]:
        reward = float(int(action) == int(self.state))
        next_state = self.rng.randrange(10)
        self.state = next_state
        next_obs = self.sample_observation(next_state)
        return next_obs, reward, next_state

## 5. Q-Network

The Q-network is a two-hidden-layer MLP over flattened MNIST images.  It outputs 10 action values, one per latent digit/action.

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, hidden_dim: int = 512, num_actions: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

## 6. Replay Buffer and Data Collection

The replay buffer stores transitions `(obs, action, reward, next_obs, state, next_state)`.  The Q-learning update uses only observations, actions, rewards, and next observations.  The latent labels are kept for diagnostics.

In [ ]:
Transition = Tuple[torch.Tensor, int, float, torch.Tensor, int, int]


class ReplayBuffer:
    def __init__(self, capacity: int):
        self.buffer: Deque[Transition] = deque(maxlen=capacity)

    def add(self, transition: Transition) -> None:
        self.buffer.append(transition)

    def sample(self, batch_size: int) -> Tuple[torch.Tensor, ...]:
        if len(self.buffer) < batch_size:
            raise ValueError(f"Replay has {len(self.buffer)} items, but batch_size={batch_size} was requested.")
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, labels, next_labels = zip(*batch)
        return (
            torch.stack(states),
            torch.tensor(actions, dtype=torch.long),
            torch.tensor(rewards, dtype=torch.float32),
            torch.stack(next_states),
            torch.tensor(labels, dtype=torch.long),
            torch.tensor(next_labels, dtype=torch.long),
        )

    def __len__(self) -> int:
        return len(self.buffer)


@torch.no_grad()
def select_action(model: QNetwork, obs: torch.Tensor, epsilon: float) -> int:
    if random.random() < epsilon:
        return random.randrange(10)
    q_values = model(obs.unsqueeze(0).to(DEVICE))
    return int(q_values.argmax(dim=1).item())


def collect_transition(env: EasyMDP, model: QNetwork, replay: ReplayBuffer, epsilon: float) -> None:
    state = int(env.state)
    obs = env.sample_observation(state)
    action = select_action(model, obs, epsilon)
    next_obs, reward, next_state = env.step(action)
    replay.add((obs.cpu(), action, reward, next_obs.cpu(), state, int(next_state)))


def populate_replay(env: EasyMDP, model: QNetwork, replay: ReplayBuffer, steps: int, epsilon: float = 1.0) -> None:
    for _ in range(steps):
        collect_transition(env, model, replay, epsilon)

## 7. Q-Learning Update and Brownian Update

The gradient-descent trajectory performs one SGD step on the Q-learning loss.  The Brownian trajectory receives a random Gaussian perturbation whose norm is matched to the SGD update norm from the gradient trajectory.

In [ ]:
def q_learning_loss(
    model: QNetwork,
    target_model: QNetwork,
    batch: Tuple[torch.Tensor, ...],
    gamma: float,
) -> torch.Tensor:
    states, actions, rewards, next_states, _, _ = batch
    states = states.to(DEVICE)
    actions = actions.to(DEVICE)
    rewards = rewards.to(DEVICE)
    next_states = next_states.to(DEVICE)

    q_sa = model(states).gather(1, actions.unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        next_q = target_model(next_states).max(dim=1).values
        td_target = rewards + gamma * next_q

    return F.mse_loss(q_sa, td_target)


def sgd_step_and_update_norm(
    model: QNetwork,
    target_model: QNetwork,
    optimizer: torch.optim.Optimizer,
    batch: Tuple[torch.Tensor, ...],
    gamma: float,
    lr: float,
) -> Tuple[float, float]:
    model.train()
    loss = q_learning_loss(model, target_model, batch, gamma)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    update_norm_sq = torch.zeros((), device=DEVICE)
    for p in model.parameters():
        if p.grad is not None:
            update_norm_sq += (lr * p.grad).pow(2).sum()

    optimizer.step()
    return float(loss.detach().cpu().item()), float(update_norm_sq.sqrt().cpu().item())


@torch.no_grad()
def brownian_step(model: QNetwork, step_norm: float) -> None:
    if step_norm <= 0:
        return

    noise = [torch.randn_like(p) for p in model.parameters()]
    total_norm = torch.sqrt(sum((n * n).sum() for n in noise))
    scale = step_norm / (float(total_norm.item()) + 1e-12)

    for p, n in zip(model.parameters(), noise):
        p.add_(n * scale)

## 8. Policy Accuracy Diagnostic

This diagnostic samples random latent states and checks whether the largest Q-value corresponds to the true latent digit.

In [ ]:
@torch.no_grad()
def evaluate_policy(model: QNetwork, env: EasyMDP, num_samples: int = 2048) -> float:
    model.eval()
    correct = 0

    for _ in range(num_samples):
        state = random.randrange(10)
        obs = env.sample_observation(state)
        pred = model(obs.unsqueeze(0).to(DEVICE)).argmax(dim=1).item()
        correct += int(pred == state)

    return correct / float(num_samples)

## 9. Probe Objective

The probe target is the current network output plus Gaussian noise.  Because the target is detached, the objective measures how easily the network can move its predictions in arbitrary local directions.

In [ ]:
def make_probe_targets(
    model: QNetwork,
    inputs: torch.Tensor,
    noise: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    model.eval()
    inputs = inputs.to(DEVICE)

    with torch.no_grad():
        outputs = model(inputs)
        if noise is None:
            noise = torch.randn_like(outputs)
        return (outputs + noise).detach(), noise.detach()


def probe_loss_from_targets(model: QNetwork, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    return F.mse_loss(model(inputs.to(DEVICE)), targets.to(DEVICE))

## 10. Gradient Covariance

For each sample, we compute the gradient of the probe loss with respect to all parameters, normalize the gradient vector, and take pairwise dot products.  The resulting matrix is a cosine similarity matrix, so its values should lie in `[-1, 1]`.

In [ ]:
def trainable_parameters(model: nn.Module) -> List[torch.nn.Parameter]:
    return [p for p in model.parameters() if p.requires_grad]


def flat_grad_from_loss(model: QNetwork, loss: torch.Tensor, create_graph: bool) -> torch.Tensor:
    grads = torch.autograd.grad(
        loss,
        trainable_parameters(model),
        create_graph=create_graph,
        retain_graph=create_graph,
    )
    return torch.cat([g.reshape(-1) for g in grads])


def sample_probe_gradient(model: QNetwork, x: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    model.zero_grad(set_to_none=True)
    loss = probe_loss_from_targets(model, x.unsqueeze(0), target.unsqueeze(0))
    grad = flat_grad_from_loss(model, loss, create_graph=False).detach().cpu()
    model.zero_grad(set_to_none=True)
    return grad


def gradient_covariance(model: QNetwork, inputs: torch.Tensor, targets: torch.Tensor, k: int) -> np.ndarray:
    model.eval()
    k = min(k, inputs.size(0))

    grads = []
    for i in tqdm(range(k), desc="gradient covariance", leave=False):
        grad = sample_probe_gradient(model, inputs[i], targets[i])
        grad = grad / grad.norm().clamp(min=1e-12)
        grads.append(grad)

    grad_matrix = torch.stack(grads)
    cov = grad_matrix @ grad_matrix.T
    return cov.numpy()


def reorder_by_kmeans(cov: np.ndarray, clusters: int = 10) -> np.ndarray:
    clusters = min(clusters, cov.shape[0])
    labels = KMeans(n_clusters=clusters, random_state=0, n_init=10).fit_predict(cov)
    order = np.argsort(labels)
    return cov[order][:, order]

## 11. Hessian ESD with Hessian-Vector Products and Lanczos

This section approximates the Hessian eigenvalue spectral density without explicitly building the full Hessian matrix.  It uses Hessian-vector products and a small Lanczos tridiagonalization.  Keep this disabled for quick runs and enable it only for smaller models or longer experiments.

In [ ]:
def make_rademacher_vector(num_params: int) -> torch.Tensor:
    v = torch.randint(0, 2, (num_params,), device=DEVICE, dtype=torch.float32)
    return 2.0 * v - 1.0


def hessian_vector_product(
    model: QNetwork,
    inputs: torch.Tensor,
    targets: torch.Tensor,
    vector: torch.Tensor,
) -> torch.Tensor:
    params = trainable_parameters(model)
    model.zero_grad(set_to_none=True)
    loss = probe_loss_from_targets(model, inputs, targets)
    grads = torch.autograd.grad(loss, params, create_graph=True)
    flat_grad = torch.cat([g.reshape(-1) for g in grads])
    grad_vector_product = torch.dot(flat_grad, vector.detach())
    hvp = torch.autograd.grad(grad_vector_product, params, retain_graph=False)
    model.zero_grad(set_to_none=True)
    return torch.cat([h.detach().reshape(-1) for h in hvp])


def hessian_esd(
    model: QNetwork,
    inputs: torch.Tensor,
    targets: torch.Tensor,
    lanczos_iter: int,
    lanczos_vectors: int,
    initial_vectors: Optional[Sequence[torch.Tensor]] = None,
    tol: float = 1e-6,
) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    num_params = sum(p.numel() for p in trainable_parameters(model))
    eigenvalues: List[np.ndarray] = []
    weights: List[np.ndarray] = []

    if initial_vectors is None:
        initial_vectors = [make_rademacher_vector(num_params) for _ in range(lanczos_vectors)]

    for initial_v in tqdm(initial_vectors, desc="Lanczos vectors", leave=False):
        v = initial_v.to(DEVICE).clone()
        v = v / v.norm()
        v_prev = torch.zeros_like(v)
        beta_prev = 0.0
        alphas: List[float] = []
        betas: List[float] = []

        for _ in tqdm(range(lanczos_iter), desc="Lanczos iterations", leave=False):
            w = hessian_vector_product(model, inputs, targets, v)
            if alphas:
                w = w - beta_prev * v_prev

            alpha = torch.dot(v, w).item()
            w = w - alpha * v
            beta = w.norm().item()
            alphas.append(alpha)

            if beta < tol:
                break

            betas.append(beta)
            v_prev = v
            v = w / beta
            beta_prev = beta

        n = len(alphas)
        tridiagonal = torch.zeros(n, n, device=DEVICE)
        for i, alpha in enumerate(alphas):
            tridiagonal[i, i] = alpha
        for i, beta in enumerate(betas[: n - 1]):
            tridiagonal[i, i + 1] = beta
            tridiagonal[i + 1, i] = beta

        vals, vecs = torch.linalg.eigh(tridiagonal)
        eigenvalues.append(vals.detach().cpu().numpy())
        weights.append((vecs[0, :] ** 2).detach().cpu().numpy())

    return np.concatenate(eigenvalues), np.concatenate(weights)


def esd_curve(
    eigenvalues: np.ndarray,
    weights: np.ndarray,
    points: int,
    sigma: Optional[float] = None,
) -> Tuple[np.ndarray, np.ndarray]:
    order = np.argsort(eigenvalues)
    eigenvalues = eigenvalues[order]
    weights = weights[order]

    lo = float(eigenvalues.min())
    hi = float(eigenvalues.max())
    width = max(hi - lo, 1e-8)
    x = np.linspace(lo - 0.05 * width, hi + 0.05 * width, points)

    if sigma is None:
        sigma = max(0.01 * width, 1e-4)

    density = np.zeros_like(x)
    normalizer = sigma * math.sqrt(2.0 * math.pi)
    for lam, weight in zip(eigenvalues, weights):
        density += weight * np.exp(-0.5 * ((x - lam) / sigma) ** 2) / normalizer

    dx = x[1] - x[0]
    density = density / (density.sum() * dx + 1e-12)
    return x, density

## 12. Snapshot Geometry

At selected target-network updates, this function computes gradient covariance and, optionally, the Hessian eigenvalue density for both the gradient-descent and Brownian trajectories.

In [ ]:
def snapshot_geometry(
    tag: str,
    gd_model: QNetwork,
    brownian_model: QNetwork,
    probe_inputs: torch.Tensor,
    config: Figure2Config,
) -> Dict[str, object]:
    probe_inputs = probe_inputs.to(DEVICE)

    # Use the same noise for both trajectories so the probe direction is comparable.
    gd_targets, shared_noise = make_probe_targets(gd_model, probe_inputs)
    brownian_targets, _ = make_probe_targets(brownian_model, probe_inputs, noise=shared_noise)

    snapshot: Dict[str, object] = {
        "tag": tag,
        "covariance": {
            "gradient": gradient_covariance(gd_model, probe_inputs, gd_targets, config.cov_batch_size),
            "brownian": gradient_covariance(brownian_model, probe_inputs, brownian_targets, config.cov_batch_size),
        },
    }

    if config.compute_hessian:
        num_params = sum(p.numel() for p in trainable_parameters(gd_model))
        lanczos_initial_vectors = [make_rademacher_vector(num_params) for _ in range(config.lanczos_vectors)]

        gd_eig, gd_weight = hessian_esd(
            gd_model,
            probe_inputs,
            gd_targets,
            config.lanczos_iter,
            config.lanczos_vectors,
            initial_vectors=lanczos_initial_vectors,
        )
        bm_eig, bm_weight = hessian_esd(
            brownian_model,
            probe_inputs,
            brownian_targets,
            config.lanczos_iter,
            config.lanczos_vectors,
            initial_vectors=lanczos_initial_vectors,
        )
        snapshot["esd"] = {
            "gradient": (gd_eig, gd_weight),
            "brownian": (bm_eig, bm_weight),
        }
        snapshot["top_eigenvalue"] = {
            "gradient": float(gd_eig.max()),
            "brownian": float(bm_eig.max()),
        }

    return snapshot

## 13. Main Experiment Loop

The two trajectories are initialized identically.  During training:

1. The gradient model collects fresh transitions and trains with SGD.
2. The target network is periodically copied from the gradient model.
3. The Brownian model receives random perturbations with the same norm as the SGD update.
4. Geometry snapshots are computed after requested target-network update counts.

In [ ]:
def run_experiment(config: Figure2Config) -> Dict[str, object]:
    set_seed(config.seed)

    dataset = load_true_label_mnist(config.data_root, config.download)
    env = EasyMDP(dataset, seed=config.seed)

    gd_model = QNetwork(config.hidden_dim).to(DEVICE)
    brownian_model = copy.deepcopy(gd_model).to(DEVICE)
    target_model = copy.deepcopy(gd_model).to(DEVICE)
    optimizer = torch.optim.SGD(gd_model.parameters(), lr=config.lr)

    replay = ReplayBuffer(config.replay_capacity)
    populate_replay(env, gd_model, replay, config.prefill_steps, epsilon=1.0)

    probe_batch = replay.sample(config.probe_batch_size)
    probe_inputs = probe_batch[0]

    logs: List[Dict[str, float]] = []
    snapshots: Dict[str, Dict[str, object]] = {}
    snapshot_targets = set(config.snapshot_target_updates)

    print(f"Device: {DEVICE}")
    print(f"Config: {asdict(config)}")

    for step in tqdm(range(1, config.train_steps + 1), desc="training"):
        for _ in range(config.online_collection_per_step):
            collect_transition(env, gd_model, replay, config.epsilon)

        batch = replay.sample(config.batch_size)
        loss, update_norm = sgd_step_and_update_norm(
            gd_model, target_model, optimizer, batch, config.gamma, config.lr
        )
        brownian_step(brownian_model, update_norm)

        if step % config.target_update_period == 0:
            target_model.load_state_dict(gd_model.state_dict())
            target_update = step // config.target_update_period
            accuracy = evaluate_policy(gd_model, env, num_samples=512)
            log_row = {
                "step": float(step),
                "target_update": float(target_update),
                "loss": float(loss),
                "update_norm": float(update_norm),
                "accuracy": float(accuracy),
            }
            logs.append(log_row)
            print(
                f"target update {target_update}: step={step}, "
                f"loss={loss:.4f}, update_norm={update_norm:.5f}, accuracy={accuracy:.3f}"
            )

            if target_update in snapshot_targets:
                tag = f"iter{target_update}"
                print(f"Computing probe geometry after target update {target_update}...")
                snapshots[tag] = snapshot_geometry(tag, gd_model, brownian_model, probe_inputs, config)

    if not snapshots:
        print("No requested snapshot was reached; computing final geometry.")
        snapshots["final"] = snapshot_geometry("final", gd_model, brownian_model, probe_inputs, config)

    return {
        "config": asdict(config),
        "logs": logs,
        "snapshots": snapshots,
        "models": {
            "gradient": gd_model,
            "brownian": brownian_model,
            "target": target_model,
        },
        "env": env,
        "probe_inputs": probe_inputs,
    }

## 14. Plotting

The top row shows Hessian ESD curves when `compute_hessian=True`.  The bottom row shows reordered gradient covariance heatmaps.  The covariance color scale is fixed to `[-1, 1]` because the entries are cosine similarities.

In [ ]:
def plot_results(results: Dict[str, object], output_path: Optional[str] = None) -> plt.Figure:
    config = Figure2Config(**results["config"])
    snapshots = results["snapshots"]
    snapshot_keys = list(snapshots.keys())
    if len(snapshot_keys) < 2:
        snapshot_keys = snapshot_keys * 2

    has_esd = "esd" in snapshots[snapshot_keys[0]]
    rows = 2 if has_esd else 1
    fig, axes = plt.subplots(rows, 4, figsize=(14, 4.2 * rows))

    if rows == 1:
        axes = np.asarray([axes])

    labels = {"gradient": "Gradient descent", "brownian": "Brownian motion"}

    if has_esd:
        for col, key in enumerate(snapshot_keys[:2]):
            ax = axes[0, col]
            for trajectory in ["gradient", "brownian"]:
                eig, weight = snapshots[key]["esd"][trajectory]
                x, density = esd_curve(eig, weight, points=config.esd_points, sigma=config.esd_sigma)
                ax.plot(x, density, label=labels[trajectory])
            ax.set_title(f"Hessian ESD: {key}")
            ax.set_xlabel("Eigenvalue")
            ax.set_ylabel("Density")
            ax.set_xlim(*config.esd_xlim)
            ax.set_ylim(*config.esd_ylim)
            ax.legend(fontsize=8)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

        axes[0, 2].axis("off")
        axes[0, 3].axis("off")
        cov_row = 1
    else:
        cov_row = 0

    cov_panels = []
    for key in snapshot_keys[:2]:
        cov_panels.extend([
            (key, "gradient", f"Gradient descent\n{key}"),
            (key, "brownian", f"Brownian motion\n{key}"),
        ])

    for col, (snap_key, trajectory, title) in enumerate(cov_panels):
        ax = axes[cov_row, col]
        cov = reorder_by_kmeans(snapshots[snap_key]["covariance"][trajectory])
        im = ax.imshow(
            cov,
            cmap="coolwarm_r",
            vmin=config.covariance_vmin,
            vmax=config.covariance_vmax,
            aspect="equal",
        )
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_ticks([config.covariance_vmin, 0.0, config.covariance_vmax])

    fig.suptitle(
        "Figure 2 reproduction: true-label MNIST MDP",
        fontsize=12,
        fontweight="bold",
    )
    fig.tight_layout()

    if output_path:
        path = Path(output_path)
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, dpi=180, bbox_inches="tight")
        print(f"Saved figure to {path}")

    return fig

## 15. Sanity Check

This cell confirms that the dataset, environment, and model connect correctly before the full experiment is run.

In [ ]:
set_seed(cfg.seed)
dataset = load_true_label_mnist(cfg.data_root, cfg.download)
env = EasyMDP(dataset, seed=cfg.seed)
model = QNetwork(cfg.hidden_dim).to(DEVICE)

obs = env.reset(state=3)
next_obs, reward, next_state = env.step(action=3)
q_values = model(obs.unsqueeze(0).to(DEVICE))

print("obs shape:", tuple(obs.shape))
print("reward:", reward, "next_state:", next_state)
print("q_values shape:", tuple(q_values.shape))

## 16. Run the Experiment

This is the main execution cell.  With the smoke-scale config, it should finish relatively quickly.  For the paper-scale version, use the next section to create a heavier config.

In [ ]:
results = run_experiment(cfg)

import pandas as pd
pd.DataFrame(results["logs"])

## 17. Plot the Result

The saved figure path is controlled by `cfg.output_path`.

In [ ]:
fig = plot_results(results, cfg.output_path)
fig

## 18. Optional: Heavier / Paper-Scale Starting Point

This cell is not executed by default.  It shows the main changes needed for a closer reproduction of Appendix A.1.  Start by enabling Hessian computation only after confirming that the smoke-scale run works on your machine.

In [ ]:
# paper_cfg_values = asdict(cfg)
# paper_cfg_values.update(PAPER_SCALE_OVERRIDES)
# paper_cfg = Figure2Config(**paper_cfg_values)
# paper_cfg
#
# paper_results = run_experiment(paper_cfg)
# plot_results(paper_results, paper_cfg.output_path)